# Frequency-search results explorer

Look at the output of an adversarial frequency search (`freq_search.py`): the
per-`(letter, pattern)` multipliers, the resulting per-word frequencies, and the
greedy strategy analyzed the *same way* as in `strategy_explorer` but under the
adversarial weighting (greedy playing against the adversarial word choices).

Generate a run first with e.g. `python math/freq_search.py 5` (saves to
`math/data/freq_search/len05.csv`); this notebook loads it back.

In [ ]:
from pathlib import Path
import sys

def find_math_dir():
    cwd = Path.cwd().resolve()
    for c in (cwd, cwd.parent, cwd.parent.parent, cwd / 'math', cwd.parent / 'math'):
        if (c / 'freq_search.py').exists():
            return c
    raise FileNotFoundError('Could not find freq_search.py. Open from the repo root, math/, or math/notebooks/.')

math_dir = find_math_dir()
if str(math_dir) not in sys.path:
    sys.path.insert(0, str(math_dir))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import freq_search, strategy

print(f'imported from {math_dir}')

## Load a search result

Loads the saved CSV for this length if present; otherwise runs one inline (which
can be slow for larger lengths).

In [ ]:
LENGTH = 5

csv_path = math_dir / 'data' / 'freq_search' / f'len{LENGTH:02d}.csv'
if csv_path.exists():
    s = freq_search.FrequencySearch.reconstruct(LENGTH, csv_path)
    print(f'loaded saved run: {csv_path.name}')
else:
    print(f'no saved run at {csv_path}\nrunning one inline (may take a while)...')
    s = freq_search.FrequencySearch(LENGTH)
    s.run(verbose=False)

game, f = s.game, s.freq
print(f'length {LENGTH}: {game.N} words, {s.M} patterns')
print(f'greedy avg misses -- uniform: {game.average_misses():.4f}   '
      f'adversarial: {game.average_misses(f):.4f}')

## Multipliers

One row per `(letter, pattern)`; `multiplier` is the tuned weight, `n_words` how
many words carry that pattern.

In [ ]:
mult_df = s.multipliers_df()
print(f'{(mult_df.multiplier == 0).sum()} patterns zeroed; '
      f'{(mult_df.multiplier > 1).sum()} boosted above 1; '
      f'max multiplier {mult_df.multiplier.max():g}')
mult_df.head(20)

## Frequencies

One row per word (frequency = product of its pattern multipliers).

In [ ]:
freq_df = s.frequencies_df()
print(f'{(freq_df.frequency == 0).sum()} words at frequency 0 (removed); '
      f'heaviest carry the adversarial weight:')
freq_df.head(15)

## Greedy: uniform vs adversarial

Each cell below runs the same analysis **both ways** — `uniform` (`freq=None`, the
`strategy_explorer` baseline) and `adversarial` (`freq=s.freq`, playing against
the tuned word distribution). The only difference is which `freq` you pass to the
`game` methods.

In [ ]:
# avg misses + opening letter, each version
for name, fr in [('uniform', None), ('adversarial', f)]:
    print(f'{name:>12}:  avg misses {game.average_misses(fr):.4f}   '
          f'opener {game.greedy_first_letter(fr)!r}')

# Mode B -- expected misses by opening letter, both versions side by side
mb = pd.DataFrame({
    'uniform':     pd.Series(game.misses_by_starting_letter(None)),
    'adversarial': pd.Series(game.misses_by_starting_letter(f)),
}).sort_values('adversarial')

x = np.arange(len(mb)); w = 0.4
fig, ax = plt.subplots(figsize=(11, 3.5))
ax.bar(x - w / 2, mb['uniform'],     w, label='uniform',     color='#3b6ea5')
ax.bar(x + w / 2, mb['adversarial'], w, label='adversarial', color='#a54b3b')
ax.set_xticks(x); ax.set_xticklabels(mb.index)
ax.set_ylabel('avg misses'); ax.legend()
ax.set_title(f'Misses by opening letter (length {LENGTH})')
plt.show()
mb.T

## Per-word difficulty (both versions)

In [ ]:
pwm_u = game.per_word_misses(None)      # uniform
pwm_a = game.per_word_misses(f)         # adversarial
words = pd.DataFrame({'word': game.words, 'freq': f,
                      'misses_uniform':     pwm_u.astype(int),
                      'misses_adversarial': pwm_a.astype(int)})

print(f'weighted avg misses -- uniform {game.average_misses(None):.4f}   '
      f'adversarial {game.average_misses(f):.4f}')

bins = range(0, int(max(pwm_u.max(), pwm_a.max())) + 2)
fig, ax = plt.subplots(figsize=(9, 3))
ax.hist(pwm_u, bins=bins, alpha=0.6, label='uniform',     color='#3b6ea5', align='left')
ax.hist(pwm_a, bins=bins, alpha=0.6, label='adversarial', color='#a54b3b', align='left')
ax.set_xlabel('misses'); ax.set_ylabel('# words'); ax.legend()
plt.show()

print('hardest words under the adversarial distribution:')
words.sort_values('misses_adversarial', ascending=False).head(15).reset_index(drop=True)

## Trace a word (both versions)

The greedy guess sequence for one word under each weighting (`+` = hit, `-` = miss).
The two can differ because the adversarial weighting changes which letters greedy
guesses first.

In [ ]:
WORD = game.words[int(pwm_a.argmax())]      # hardest adversarial word; or set WORD = 'bench'

for name, fr in [('uniform', None), ('adversarial', f)]:
    hits = game.trace(WORD, fr)
    seq = '  '.join(f'{l}{"+" if h else "-"}' for l, h in hits)
    print(f'{name:>12}  ({sum(1 for _, h in hits if not h)} misses):  {seq}')